In [1]:
import time

notebook_start_time = time.time()

# Set up environment

In [2]:
import sys
from pathlib import Path


def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False


def clone_repository() -> None:
    !git clone https://github.com/decodingml/hands-on-recommender-system.git
    %cd hands-on-recommender-system/


def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml


if is_google_colab():
    clone_repository()
    install_dependencies()

    root_dir = str(Path().absolute())
    print("⛳️ Google Colab environment")
else:
    root_dir = str(Path().absolute().parent)
    print("⛳️ Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    print(f"Adding the following directory to the PYTHONPATH: {root_dir}")
    sys.path.append(root_dir)

⛳️ Local environment
Adding the following directory to the PYTHONPATH: c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course


# Online inference pipeline: Deploying and testing the real-time ML services

In this notebook, we will dig into the inference pipeline and deploy it to Hopsworks as a real-time service.

## Imports

In [3]:
import warnings

warnings.filterwarnings("ignore")

from loguru import logger

from recsys import hopsworks_integration

from recsys.config import settings

## Connect to Hopsworks Feature Store </span>

In [4]:
import hopsworks

hopsworks_host = "eu-west.cloud.hopsworks.ai"

project = hopsworks.login(
    host=hopsworks_host,
    api_key_value=settings.HOPSWORKS_API_KEY.get_secret_value()
)

# Thêm logic này để tự tạo project nếu tài khoản bị trống
if project is None:
    project = hopsworks.create_project("recommender_system", description="H&M Recommender")

fs = project.get_feature_store()

2026-03-11 02:13:00,406 INFO: Initializing external client
2026-03-11 02:13:00,406 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-03-11 02:13:03,430 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31873


# Deploying the ranking inference pipeline


You start by deploying your ranking model. Since it is a CatBoost model you need to implement a `Predict` class that tells Hopsworks how to load the model and how to use it:

In [5]:
ranking_deployment = hopsworks_integration.ranking_serving.HopsworksRankingModel.deploy(
    project=project
)

Uploading c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course\recsys\inference\ranking_transformer.py: 100.000%|██████████| 4635/4635 elapsed<00:01 remaining<00:00
Uploading c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course\recsys\inference\ranking_predictor.py: 100.000%|██████████| 1151/1151 elapsed<00:01 remaining<00:00


Deployment created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/31873/deployments/8200
Before making predictions, start the deployment by using `.start()`


Now, we have to explicitly start the deployment:

In [6]:
ranking_deployment.start()

Deployment is ready: 100%|██████████| 6/6 [00:38<00:00,  6.47s/it]    

Start making predictions by using `.predict()`


In [7]:
# Check logs in case of failure
# ranking_deployment.get_logs(component="transformer", tail=200)

## Test the ranking inference pipeline


In [8]:
def get_top_recommendations(ranked_candidates, k=3):
    return [candidate[-1] for candidate in ranked_candidates["ranking"][:k]]

Let's define a dummy test example to test our ranking deployment (only the `customer_id` has to match):

In [9]:
test_ranking_input = [
        {
            "customer_id": "d327d0ad9e30085a436933dfbb7f77cf42e38447993a078ed35d93e3fd350ecf",
            "month_sin": 1.2246467991473532e-16,
            "query_emb": [
                0.214135289,
                0.571055949,
                0.330709577,
                -0.225899458,
                -0.308674961,
                -0.0115124583,
                0.0730511621,
                -0.495835781,
                0.625569344,
                -0.0438038409,
                0.263472944,
                -0.58485353,
                -0.307070434,
                0.0414443575,
                -0.321789205,
                0.966559,
            ],
            "month_cos": -1.0,
        }
    ]

# Test ranking deployment
ranked_candidates = ranking_deployment.predict(inputs=test_ranking_input)

# Retrieve article ids of the top recommended items
recommendations = get_top_recommendations(ranked_candidates["predictions"], k=3)
recommendations

['757613003', '856667006', '608776020']

Check logs in case of failure:

In [10]:
# ranking_deployment.get_logs(component="transformer", tail=200)

# Deploying the query inference pipeline

In [11]:
query_model_deployment = (
    hopsworks_integration.two_tower_serving.HopsworksQueryModel.deploy(ranking_model_type="ranking")
)

2026-03-11 02:14:06,356 INFO: Closing external client and cleaning up certificates.
2026-03-11 02:14:06,372 INFO: Connection closed.
2026-03-11 02:14:06,382 INFO: Initializing external client
2026-03-11 02:14:06,384 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-03-11 02:14:13,156 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31873
2026-03-11 02:14:16,961 INFO: Closing external client and cleaning up certificates.
2026-03-11 02:14:16,975 INFO: Connection closed.
2026-03-11 02:14:16,980 INFO: Initializing external client
2026-03-11 02:14:16,983 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-03-11 02:14:41,289 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31873
Secret created successfully, explore it at https://eu-west.cloud.hopsworks.ai:443/account/secrets


Uploading c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course\recsys\inference\query_transformer.py: 100.000%|██████████| 3169/3169 elapsed<00:01 remaining<00:00


Deployment created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/31873/deployments/8201
Before making predictions, start the deployment by using `.start()`


At this point, you have registered your deployment. To start it up you need to run:

In [12]:
query_model_deployment.start()

Deployment is ready: 100%|██████████| 6/6 [00:33<00:00,  5.62s/it]    

Start making predictions by using `.predict()`


##  Testing the inference pipeline </span>

Define a test input example:

In [14]:
data = [
    {
        "customer_id": "d327d0ad9e30085a436933dfbb7f77cf42e38447993a078ed35d93e3fd350ecf",
        "transaction_date": "2022-11-15T12:16:25.330916",
    }
]

Test out the deployment:

In [15]:
ranked_candidates = query_model_deployment.predict(inputs=data)

# Retrieve article ids of the top recommended items
recommendations = get_top_recommendations(ranked_candidates["predictions"], k=3)
recommendations

['608776020', '485548007', '887200003']

Check logs in case of failure:

# Stopping the Hopsworks deployments </span>

Stop the deployment when you're not using it.

In [17]:
ranking_deployment.stop()
query_model_deployment.stop()

Deployment is stopped: 100%|██████████| 4/4 [00:11<00:00,  2.89s/it]        


## Inspecting the deployments in Hopsworks UI </span>

View results in [Hopsworks Serverless](https://rebrand.ly/serverless-github): **Data Science → Deployments**

---

In [18]:
notebook_end_time = time.time()
notebook_execution_time = notebook_end_time - notebook_start_time

logger.info(
    f"⌛️ Notebook Execution time: {notebook_execution_time:.2f} seconds ~ {notebook_execution_time / 60:.2f} minutes"
)

2026-03-11 02:15:58.224 | INFO     | __main__:<module>:4 - ⌛️ Notebook Execution time: 204.97 seconds ~ 3.42 minutes
